# OCR Project - CRNN + CTC Pipeline
This notebook demonstrates how to train and use the new CRNN model for character recognition.

In [ ]:
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from ocr_model import (
    train, build_ocr_model, preprocess_image, decode_prediction, 
    load_dataset_from_char_folders, build_tf_dataset, 
    IMG_WIDTH, IMG_HEIGHT, VOCAB_SIZE, CHARACTERS
)

# 1. GPU Configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"Using {len(gpus)} GPU(s)")
else:
    print("Using CPU")

## 1. Data Loading & Visualization

In [ ]:
DATASET_DIR = "DATASET"
image_paths, labels = load_dataset_from_char_folders(DATASET_DIR)

# Build a small dataset for visualization
viz_ds = build_tf_dataset(image_paths[:100], labels[:100], batch_size=8, shuffle=True)

for inputs, targets in viz_ds.take(1):
    images = inputs['image']
    plt.figure(figsize=(15, 5))
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        # Transpose back (W, H, 1) -> (H, W)
        img = images[i].numpy().transpose(1, 0, 2).squeeze()
        plt.imshow(img, cmap='gray')
        plt.title(f"Char: {labels[i]}")
        plt.axis('off')
    plt.show()

## 2. Model Summary

In [ ]:
training_model, prediction_model = build_ocr_model(IMG_WIDTH, IMG_HEIGHT, VOCAB_SIZE)
training_model.summary()

## 3. Full Training
Run this cell to start the training process. The progress bar will show the training status for each epoch.

In [ ]:
# Start training and capture history
history, prediction_model = train(
    dataset_dir=DATASET_DIR,
    checkpoint_dir="checkpoints",
    use_char_folders=True
)

## 4. Training Progress Visualization
Visualize the loss curves to monitor convergence.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss', color='#4CAF50', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', color='#FF5722', linewidth=2)
plt.title('OCR Model - CTC Loss Progress', fontsize=14, pad=15)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 5. Inference Test

In [ ]:
def test_prediction(img_path):
    img = preprocess_image(img_path)
    img = np.expand_dims(img, axis=0)
    pred = prediction_model.predict(img, verbose=0)
    decoded = decode_prediction(pred[0])
    
    plt.imshow(img[0].transpose(1, 0, 2).squeeze(), cmap='gray')
    plt.title(f"Predicted: {decoded}")
    plt.axis('off')
    plt.show()

# Test on a sample from the dataset
if len(image_paths) > 0:
    test_prediction(image_paths[0])

In [ ]:
def test_prediction(img_path):
    img = preprocess_image(img_path)
    img = np.expand_dims(img, axis=0)
    pred = prediction_model.predict(img, verbose=0)
    decoded = decode_prediction(pred[0])
    
    plt.imshow(img[0].transpose(1, 0, 2).squeeze(), cmap='gray')
    plt.title(f"Predicted: {decoded}")
    plt.axis('off')
    plt.show()

# Replace with a path to one of your images
# test_prediction(image_paths[0])